# Preprocesamiento de datos: rotación temprana de operarios

Limpieza e ingeniería de variables de las bases de ingresos y ceses de una empresa industrial (proyecto ESAN, People Analytics).

**Los datos no se incluyen en este repositorio** porque contienen información personal de trabajadores reales.


In [ ]:
import pandas as pd, re, unicodedata
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path
from google.colab import files
import datetime


In [ ]:
df_ceses = pd.read_excel("data/ingresos_ceses.xlsx", sheet_name=1)
df_ingresos = pd.read_excel("data/ingresos_ceses.xlsx", sheet_name=0)
pd.options.display.max_rows = 500

# 1. Vista rapida de las features

In [ ]:
df_ceses.info()

In [ ]:
df_ingresos.info()

# 2. Limpieza de Datos

## Eliminar columnas innecesarias en el modelo de machine learning

In [ ]:
## DATA DE CESES

df_ceses.columns = df_ceses.columns.str.strip()  # quita espacios en los nombres
cols_a_quitar = ["Cod. Trabajador", "Nom. Trabajador", "EMAIL", "Dir. Trabajador", "DEPARTAMENTO"]
df_ceses = df_ceses.drop(columns=cols_a_quitar, errors="ignore")
df_ceses = df_ceses.rename(columns= {"E.S": "E.C", 'Grado de Estudios': 'Grado Educativo'})

In [ ]:
## DATA DE INGRESOS

df_ingresos.columns = df_ingresos.columns.str.strip()  # quita espacios en los nombres
cols_a_quitar = ["Cod. Trabajador", "Nom. Trabajador", "EMAIL", "Dir. Trabajador", "DEPARTAMENTO"]
df_ingresos = df_ingresos.drop(columns=cols_a_quitar, errors="ignore")

In [ ]:
## Vistazo rapido a las tablas

df_ceses.head(3)

In [ ]:
## Vistazo rapido a las tablas

df_ingresos.head(3)

## Eliminar duplicados por ultima fecha de ingreso o cese

In [ ]:
def ultimo_puesto_por_ingreso(
    df,
    id_col='DNI',
    fecha_ingreso_col='FECHA_INGRESO',
    fecha_cese_col='FECHA_CESE',
    dayfirst=True
):
    # Copia de trabajo
    out = df.copy()

    # Limpieza ligera de nombres por si vienen con espacios
    out.columns = out.columns.str.strip()

    # Parseo de fechas robusto
    out[fecha_ingreso_col] = pd.to_datetime(out[fecha_ingreso_col], dayfirst=dayfirst, errors='coerce')
    if fecha_cese_col in out.columns:
        out[fecha_cese_col] = pd.to_datetime(out[fecha_cese_col], dayfirst=dayfirst, errors='coerce')

    # Quitamos filas sin fecha de ingreso (no se puede determinar su tiempo de permanencia, clave en nuestro modelo de machine learning)
    out = out.dropna(subset=[fecha_ingreso_col])

    # Orden: por persona, luego por fecha de ingreso (↑) y como desempate por fecha de cese (↑).
    # Luego nos quedamos con la última fila por persona.
    sort_cols = [id_col, fecha_ingreso_col]
    if fecha_cese_col in out.columns:
        sort_cols.append(fecha_cese_col)

    out = (
        out.sort_values(sort_cols, ascending=True, na_position='first')
           .groupby(id_col, as_index=False)
           .tail(1)
           .sort_values(id_col)
           .reset_index(drop=True)
    )
    return out

In [ ]:
df_prueba = ultimo_puesto_por_ingreso(df_ceses, id_col="DNI Trabajador", fecha_ingreso_col='Fec. Ingreso', fecha_cese_col='Fec. Cese', dayfirst=True)

In [ ]:
df_prueba.head(3)

In [ ]:
def dedup_por_ultimo_ingreso(df, id_col='DNI Trabajador', fecha_col='Fec. Ingreso', dayfirst=True):
    out = df.copy()
    # Parseo de fecha (robusto)
    out[fecha_col] = pd.to_datetime(out[fecha_col], dayfirst=dayfirst, errors='coerce')

    # Ordenar SOLO por fecha de ingreso (NaT al final)
    out = out.sort_values(fecha_col, ascending=True, na_position='first')

    # Quitar reingresos: mantener el último por persona (el más reciente por la ordenación anterior)
    out = out.drop_duplicates(subset=[id_col], keep='last')

    # Presentación final: seguir orden SOLO por fecha de ingreso
    out = out.sort_values(fecha_col, ascending=True, na_position='last').reset_index(drop=True)
    return out

# Aplicación a la tabla de ingresos
df_limpio = dedup_por_ultimo_ingreso(df_ingresos, id_col='DNI Trabajador', fecha_col='Fec. Ingreso', dayfirst=True)

In [ ]:
df_limpio.head(3)

# Union de tablas de ceses junto a ingresos

In [ ]:

def unir_por_id(
    df_base: pd.DataFrame,
    df_cesados: pd.DataFrame,
    id_col: str = 'DNI Trabajador',
    prefer: str = 'base',                   # 'base' o 'cesados'
    dayfirst: bool = True,
    ascending: bool = True,
) -> pd.DataFrame:
    a = df_base.copy()
    b = df_cesados.copy()

    # Normalizar ID
    a[id_col] = a[id_col].astype(str).str.strip()
    b[id_col] = b[id_col].astype(str).str.strip()

    # FULL OUTER JOIN
    out = a.merge(b, on=id_col, how='outer', suffixes=('', '_ces'))

    # Columnas que existen en ambos (excepto el ID)
    overlap = [c for c in a.columns if c in b.columns and c != id_col]

    # Consolidar duplicadas: prioriza 'base' o 'cesados'
    for col in overlap:
        ces = f'{col}_ces'
        if ces in out.columns:
            if prefer == 'base':
                out[col] = out[col].combine_first(out[ces])
            else:  # prefer == 'cesados'
                out[col] = out[ces].combine_first(out[col])
            out.drop(columns=[ces], inplace=True)

    return out


In [ ]:
df_out = unir_por_id(df_limpio, df_prueba)

# Feature Engineering

In [ ]:
df_out.info()

In [ ]:
df_out['PROVINCIA'] = df_out['PROVINCIA'].fillna("LIMA")

In [ ]:
df_out = df_out.drop(columns=["MOT. CESE", "DISTRITO"])

In [ ]:
df_out['SEXO'] = df_out['SEXO'].apply(lambda x: 1 if x == "MASCULINO" else 0)
df_out['Fec. Nacimiento'] = pd.to_datetime(df_out['Fec. Nacimiento'], errors='coerce', dayfirst=True)
hoy = pd.Timestamp.now(tz='America/Lima').normalize().tz_localize(None)
# 3) Componentes de la fecha de nacimiento
b_y = df_out['Fec. Nacimiento'].dt.year
b_m = df_out['Fec. Nacimiento'].dt.month
b_d = df_out['Fec. Nacimiento'].dt.day

# 4) ¿Aún no cumplió años este año?
no_cumplio = (b_m > hoy.month) | ((b_m == hoy.month) & (b_d > hoy.day))

# 5) Edad entera (nullable Int64 para permitir NA)
df_out['Edad'] = (hoy.year - b_y - no_cumplio.astype('Int64')).astype('Int64')

# df_out = df_out.drop(columns=['Fec. Nacimiento'])

In [ ]:
df_out['Est. Asignacion'] = df_out['Est. Asignacion'].apply(lambda x: 1 if x == "SI" else 0)
df_out['Est. Trabajador'] = df_out['Est. Trabajador'].apply(lambda x: 1 if x == "CESADO" else 0)

In [ ]:
df_out.info()

In [ ]:
def norm(s):
    if pd.isna(s):
        return ""
    s = str(s).lower().strip()
    # quitar acentos
    s = ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')
    # limpiar símbolos y espacios
    s = re.sub(r'[^a-z0-9\s]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

df_out['Cargo'] = df_out['Cargo'].apply(norm)

def nivel_por_cargo_norm(s):
    if not s:  # vacío -> categoría por defecto
        return 'Otro'
    if re.search(r'\b(supervisor|lider)\b', s):   return 'Jefatura/Supervision'
    if re.search(r'\b(analista|inspector)\b', s): return 'Analista/Inspector'
    if re.search(r'\b(tecnic|electric|electron)\b', s): return 'Tecnico'
    if re.search(r'\b(asistente|auxiliar)\b', s): return 'Soporte'
    if re.search(r'\b(operador|maquinista)\b', s): return 'Operador'
    if re.search(r'\b(operario|soldador|torno|arenador|oxicorte)\b', s): return 'Operario'
    if re.search(r'\b(conductor|chofer)\b', s):  return 'Conductor'
    return 'Otro'

df_out['Nivel_Rol'] = df_out['Cargo'].apply(nivel_por_cargo_norm)

In [ ]:
# Fecha de corte (Lima)
t0 = pd.Timestamp.now(tz='America/Lima').normalize().tz_localize(None)

# Fechas a datetime
df_out['Fec. Ingreso'] = pd.to_datetime(df_out['Fec. Ingreso'], errors='coerce', dayfirst=True)
df_out['Fec. Cese']    = pd.to_datetime(df_out['Fec. Cese'],    errors='coerce', dayfirst=True)

fi = df_out['Fec. Ingreso']
fc = df_out['Fec. Cese']

# Fin del intervalo base: cesados antes/en t0 -> Cese; resto -> t0
end = fc.where(fc.notna() & (fc <= t0), t0)

# Condición extra: si Ingreso > Cese, forzar fin = t0 (hoy - Ingreso)
wrong_order = fi.notna() & fc.notna() & (fi > fc)
end = end.where(~wrong_order, t0)

# Meses transcurridos (resta 1 si no alcanzó el "día" del mes final)
ant_meses = (
    (end.dt.year - fi.dt.year) * 12
  + (end.dt.month - fi.dt.month)
  - (end.dt.day < fi.dt.day).astype('Int64')
)

# Evitar valores inválidos (ingreso nulo o posterior al fin)
df_out['ANTIGUEDAD_MESES'] = ant_meses.where(fi.notna() & (fi <= end)).astype('Int64')


In [ ]:
df_out.info()

In [ ]:
df_out = df_out.drop(columns=['Cargo'])

In [ ]:
df_out = df_out.drop(columns=['Fec. Cese', 'Fec. Ingreso'])

In [ ]:
df_out = df_out.join(pd.get_dummies(df_out['Division'], prefix="division")).drop("Division", axis=1)
df_out = df_out.join(pd.get_dummies(df_out['Nivel_Rol'], prefix="nivel_rol")).drop("Nivel_Rol", axis=1)
df_out = df_out.join(pd.get_dummies(df_out['E.C'], prefix="estado_civil")).drop("E.C", axis=1)
df_out = df_out.join(pd.get_dummies(df_out['Grado Educativo'], prefix="grado_educativo")).drop("Grado Educativo", axis=1)


In [ ]:
df_out = df_out.join(pd.get_dummies(df_out['PROVINCIA'], prefix="provincia")).drop("PROVINCIA", axis=1)

In [ ]:
df_out = df_out.map(lambda x: 1 if x is True else 0 if x is False else x)

In [ ]:
df_out = df_out.drop('DNI Trabajador', axis=1)

In [ ]:
df_out = df_out.drop('Fec. Nacimiento', axis=1)

In [ ]:
df_out.info()

In [ ]:
df_out.to_excel("dataset_randomforest.xlsx", index=False)
files.download("dataset_randomforest.xlsx")